# Notebook 2: training and evaluation

This notebook trains Sentinel-1 damage classifiers from the patches created
by notebook 1. All inputs are read from Drive; Earth Engine is not required.
Each experiment stores its configuration, checkpoints, predictions, metrics
and figures in `sentinel_sar/experiments/{experiment_name}/`.

The first-stage CNN receives pre-event and post-event VV and VH patches. The
same CNN can also score post-event imagery at temporal offsets. Optional
second-stage models combine the CNN score with spatial neighbourhood features
and, when enabled, the temporal score series.

Validation compares the following prediction variants:

| variant | additional information |
|---|---|
| `cnn` | patch classifier only |
| `cnn+fixed_neighbor_rule` | fixed weighted-neighbour control |
| `xgb_spatial` | learned spatial second stage |
| `xgb_spatial_temporal` | learned spatial and temporal second stage |

The hand-set and Optuna-tuned CNN runs are compared using mean ROC AUC
across validation dates. One threshold is then fitted to the pooled
validation predictions of the selected candidate. Only that frozen pipeline
is evaluated on the Gaza test band and the city-level holdouts.

| decision | fitted/selected on | excluded data |
|---|---|---|
| CNN weights (gradient updates) | train band | stack, validation, test and holdouts |
| CNN early stopping (best-epoch selection) | validation band | test and holdouts |
| CNN hyperparameters | validation band | test and holdouts |
| second-stage model | stack band | test and holdouts |
| second-stage early stopping | validation band | test and holdouts |
| final candidate and threshold | validation band | test and holdouts |

Validation therefore never contributes a gradient update, but it does decide
*which epoch's* trained weights are kept, for both the CNN (early stopping)
and the second stage.


## 1. Setup and configuration


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q optuna xgboost torchinfo

In [ ]:
import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/War-Damage-Detection"
SENTINEL_DIR = os.path.join(PROJECT_ROOT, "sentinel_sar")
for path in (PROJECT_ROOT, SENTINEL_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

import json
import pickle
import copy
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from xgboost import XGBClassifier
from sklearn.metrics import (precision_recall_fscore_support, auc,
                             roc_auc_score, precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)

from sentinel_sar.pipeline import (CITY_REGISTRY, PatchSource,
                      EXPERIMENTS_DIR,
                      load_city, usable_mask, city_source,
                      wanted_channels, label_matrix, propagate_labels,
                      parse_split_entry, parse_entry, expand_split, strip_dates,
                      band_mask, split_assignment,
                      temporal_offsets, temporal_window_days, shift_date,
                      spatial_smooth, neighbour_features, temporal_features,
                      MODEL_REGISTRY, build_model, predict_probs,
                      best_f1_threshold, experiment_dirs,
                      city_role, held_out_cities, holdout_split_entries,
                      check_split_holdout, load_run, list_saved_runs,
                      list_experiments)

# ---------------------------------------------------------------------------
# EVALUATE_ONLY: reload a finished experiment instead of fitting anything.
#
# False: train or resume, run HPO, fit the stackers and select a model.
# True: load the CNNs and stackers already saved for this experiment and go
#        straight to scoring. Nothing is fitted, so nothing can change: the
#        frozen final_selection.json still decides the pipeline and the
#        threshold. Validation and any unscored holdout cities can then be
#        processed after a Colab restart without fitting.
#
# In this mode mu/sd come from the checkpoint rather than being recomputed.
# They are part of the fitted model and must remain unchanged.
# fitted model.
# ---------------------------------------------------------------------------
EVALUATE_ONLY = False

if not os.path.exists(EXPERIMENTS_DIR):
    print("no experiments folder yet")
else:
    _known = list_experiments()
    if len(_known):
        print("experiments on Drive:")
        print(_known.to_string(index=False))
        print()

CONFIG = {
    "experiment_name": "exp001_base_cnn_sar_temporal_roc_auc_compact_xgb",
    "seed": 0,
    "selection_metric": "roc_auc",

    # Select the event phases supplied to the model. Each enabled phase
    # contains Sentinel-1 VV and VH channels.
    "features": {
        "pre_event": True,
        "post_event": True,
    },

    # Any architecture registered in MODEL_REGISTRY can be selected here.
    "model": "base_cnn",

    # Architecture hyperparameters. Optuna searches these; these values are
    # the hand-set baseline it is compared against.
    # width is the first block's filter count, and every later block doubles it.
    # depth is the number of conv blocks. Each block is a 3 x 3 convolution,
    # BatchNorm, ReLU and MaxPool(2), so it halves the spatial size.
    "model_params": {},

    # A split entry is "City", or "City:lo-hi" for a latitude quantile band,
    # optionally followed by "@date". "@*" expands to every registered
    # assessment date of that city.
    #
    # Bands make train, validation and test spatially disjoint: a random split
    # would leak through spatial autocorrelation, because a 32 px patch is
    # 320 m across and neighbouring buildings share pixels.
    #
    # Validation and stack use every registered assessment date. This supports
    # a date-agnostic threshold while retaining spatial separation. Repeated
    # buildings across dates are not independent observations, so the effective
    # sample size is smaller than the row count.
    #
    # Holding out a date would retain most of the same buildings across roles.
    # Spatial bands avoid this direct building overlap.
    "split": {
        "train": ["Gaza:0.55-1.00@*"],              # explicitly use all registered dates
        # The second stage needs CNN scores from buildings excluded from CNN
        # training, so it uses a separate band rather than train or validation.
        "stack": ["Gaza:0.44-0.55@*"],
        "val":   ["Gaza:0.33-0.44@*"],
        "test":  ["Gaza:0.00-0.33@20240503",
                  "Gaza:0.00-0.33@20240706",
                  "Gaza:0.00-0.33@20240906"],

        # Whole cities excluded from every development band above. The test
        # split measures unseen ground within Gaza, while these entries measure
        # transfer to cities with different imagery and building stock.
        # Results are reported separately from the in-city test band.
        #
        # Defaults to every role="holdout" city in CITY_REGISTRY, so
        # registering one there is enough. Replace with an explicit list to
        # evaluate only some of them.
        "holdout": holdout_split_entries(),
    },

    # If True, enforce cumulative labels: once damaged, remain damaged at
    # subsequent assessments. The @date / @* syntax controls which dates
    # enter training; this flag only controls label correction.
    "label_temporal": False,

    # Fixed-weight smoothing of the CNN's scores, used as the control variant.
    # Not applied to the CNN's own scores anywhere else: the base CNN should
    # judge each patch on its own and leave the neighbourhood to stage two.
    "spatial_smoothing": {"k": 8, "weight": 0.3},

    # Second stage over the CNN's scores.
    "stacking": {
        "enabled": True,
        "ks": [8, 32],            # neighbourhood sizes, in buildings
        "spatial_features": [
            "score_t0", "k8_mean", "k8_std",
            "k32_mean", "k32_std",
        ],
        "temporal_features": [
            "score_t-24", "score_t-12", "score_t+12", "score_t+24",
            "pre_change", "post_persistence",
        ],
        "n_estimators": 600,
        "max_depth": 4,
        "learning_rate": 0.05,
        "early_stopping_rounds": 40,
    },

    # Temporal features for the second stage. offsets None = PREP["temporal"].
    # Set enabled False to skip scoring the offset dates entirely.
    "temporal_stacking": {
        "enabled": True,
        "offsets": [-24, -12, 0, 12, 24],
    },

    # Hyperparameter search over the CNN. Trials are short and run on a
    # subsample; the winner is then retrained with the full budget.
    "hpo": {
        "enabled": True,
        "n_trials": 10,            # target total; may be increased when resuming
        "max_epochs": 25,          # per trial, vs max_epochs below for the final fit
        "train_subsample": 0.5,    # fraction of training rows per trial
        "timeout_minutes": None,  # None for no limit
        "patience": 6,
    },

    # training
    "batch_size": 128,
    "learning_rate":  1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 120,
    "early_stopping_patience": 10,
    "num_workers": 2,
    "predict_batch": 1024,
}

# Console progress only; changing this does not define a new experiment.
HPO_LOG_EVERY = 5


def validate_config(cfg):
    """Fail early with a readable message instead of a confusing crash later."""
    f = cfg["features"]
    if not (f["pre_event"] or f["post_event"]):
        raise ValueError("enable at least one of pre_event / post_event")
    if cfg["model"] not in MODEL_REGISTRY:
        raise ValueError(f"unknown model, choose one of {list(MODEL_REGISTRY)}")
    if cfg["model"] == "siamese" and not (f["pre_event"] and f["post_event"]):
        raise ValueError("siamese compares pre and post, enable both")
    if cfg["temporal_stacking"]["enabled"] and not f["post_event"]:
        raise ValueError("temporal features vary the POST window, so post_event "
                         "must be on for them to mean anything")
    if cfg["temporal_stacking"]["enabled"] and not cfg["stacking"]["enabled"]:
        raise ValueError("temporal features are consumed by the second stage; "
                         "enable stacking or disable temporal_stacking")
    if tuple(cfg["stacking"]["ks"]) != (8, 32):
        raise ValueError("the compact spatial schema requires ks=[8, 32]")
    if (cfg["temporal_stacking"]["enabled"] and
            tuple(cfg["temporal_stacking"]["offsets"]) !=
            (-24, -12, 0, 12, 24)):
        raise ValueError(
            "the compact temporal schema requires offsets "
            "[-24, -12, 0, 12, 24]")


def validate_split(cfg):
    """No holdout city may leak into a part the model is developed on."""
    leaks = check_split_holdout(cfg["split"])
    if leaks:
        for part, entry, city in leaks:
            print(f"   {part}: {entry}  (city {city} is role=holdout)")
        raise RuntimeError(
            "holdout cities appear in a development split part (listed "
            "above). Move them to split['holdout'], or change the city's "
            "role in CITY_REGISTRY if it really is meant to be developed on.")


validate_config(CONFIG)
validate_split(CONFIG)
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

SPLIT = expand_split(CONFIG["split"])
OFFSETS = (CONFIG["temporal_stacking"]["offsets"] or temporal_offsets()
           if CONFIG["temporal_stacking"]["enabled"] else [0])
if 0 not in OFFSETS:
    OFFSETS = sorted(set(OFFSETS) | {0})

def experiment_identity(cfg):
    """The config that defines the FITTED pipeline.

    Two things are excluded because changing them cannot change any model,
    any selection or any threshold:

    hpo.n_trials / hpo.timeout_minutes
        resumable budgets - more trials continue the same study.

    split["holdout"]
        whole cities that are only ever scored, never fitted on. Registering
        a new holdout city must not invalidate a finished experiment, and it
        also means experiments created before the holdout part existed still
        compare equal.
    """
    identity = copy.deepcopy(cfg)
    # Older experiment configs may contain sensor switches. The current
    # pipeline is S1-only, so experiment identity depends only on the two
    # event-phase switches. This preserves semantically identical runs.
    features = identity.get("features", {})
    identity["features"] = {
        key: features.get(key, True) for key in ("pre_event", "post_event")
    }
    # .get(): this also runs against configs embedded in older checkpoints,
    # which may predate a key entirely.
    identity.get("hpo", {}).pop("n_trials", None)
    identity.get("hpo", {}).pop("timeout_minutes", None)
    identity.get("split", {}).pop("holdout", None)
    return identity


def atomic_config_dump(cfg, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(cfg, fh, indent=2)
    os.replace(tmp, path)


EXP_ROOT, OUT = experiment_dirs(CONFIG["experiment_name"])
config_path = os.path.join(EXP_ROOT, "config.json")

if os.path.exists(config_path):
    with open(config_path) as fh:
        previous_config = json.load(fh)

    if experiment_identity(previous_config) != experiment_identity(CONFIG):
        raise RuntimeError(
            f"{EXP_ROOT} already contains a different scientific configuration. "
            "Only hpo.n_trials and hpo.timeout_minutes may change in place; "
            "otherwise choose a new experiment_name."
        )

    old_trials = int(previous_config["hpo"]["n_trials"])
    new_trials = int(CONFIG["hpo"]["n_trials"])
    if new_trials < old_trials:
        raise RuntimeError(
            f"Cannot reduce hpo.n_trials from {old_trials} to {new_trials} "
            "inside an existing experiment."
        )

    budget_changed = (
        old_trials != new_trials
        or previous_config["hpo"].get("timeout_minutes") !=
        CONFIG["hpo"].get("timeout_minutes"))
    holdout_changed = (
        previous_config.get("split", {}).get("holdout") !=
        CONFIG["split"].get("holdout"))
    if budget_changed and os.path.exists(
            os.path.join(OUT["metrics"], "final_selection.json")):
        raise RuntimeError(
            "The final selection is already frozen. Start a new experiment "
            "before expanding or changing the HPO budget.")

    if previous_config != CONFIG:
        atomic_config_dump(CONFIG, config_path)
        if budget_changed:
            print(
                f"Updated the resumable HPO budget: n_trials {old_trials} -> "
                f"{new_trials}. Existing completed/pruned trials will be reused."
            )
        if holdout_changed:
            print(
                "Updated the holdout city list. These are evaluation targets "
                "only, so every fitted model and the frozen selection are "
                "unaffected; Section 11 will score the cities it has not "
                "already sealed."
            )
    else:
        print(
            "Existing compatible configuration found. Completed models and HPO "
            "trials will be reused; unfinished CNN training will resume."
        )
else:
    atomic_config_dump(CONFIG, config_path)
    print("Created new experiment configuration.")

CHANNEL_NAMES = wanted_channels(CONFIG["features"])
print(f"experiment folder: {EXP_ROOT}")
print(f"device: {device}")
print(f"active channels ({len(CHANNEL_NAMES)}): {CHANNEL_NAMES}")
print(f"temporal offsets scored: {OFFSETS}")
print(f"mode: {'EVALUATE_ONLY - nothing will be fitted' if EVALUATE_ONLY else 'train / fit'}")
print("\nsplit after expanding @*:")
for part in ["train", "stack", "val", "test"]:
    print(f"  {part:8s} {SPLIT.get(part, [])}")
print(f"  {'holdout':8s} {SPLIT.get('holdout', [])}")
print(f"\nholdout cities registered: {held_out_cities()}")
print("test = Gaza's own southern band (unseen ground, known city); "
      "holdout = whole unseen cities")

## 2. Build the development sets

Each split entry becomes one part: one latitude band, city and assessment
date. A part stores row indices into the canonical building table, while pixel
arrays remain memory mapped and are loaded by batch.

Rows require valid pre-event and labelled-date post-event patches. Missing
patches at temporal offsets remain aligned and produce NaN scores. XGBoost
handles these missing feature values natively.

Only train, stack and validation parts are constructed in this section. Test
parts are constructed after the final validation selection has been frozen.


In [ ]:
CITY_CACHE = {}


def city_data(name):
    if name not in CITY_CACHE:
        CITY_CACHE[name] = load_city(name)
    return CITY_CACHE[name]


def labels_for(d, date, propagate):
    """The label column for one date, optionally propagated forward in time."""
    dates = CITY_REGISTRY[d["city"]]["label_dates"]
    if not propagate or len(dates) < 2:
        return d["table"][f"class_{date}"].to_numpy(int)
    Y = propagate_labels(label_matrix(d["table"], dates))
    return Y[:, dates.index(date)]


def make_part(entry, propagate=False):
    """One band of one city at one date, as row indices into its table."""
    cty, lo, hi, date = parse_entry(entry)
    d = city_data(cty)
    date = date or CITY_REGISTRY[cty]["label_dates"][-1]
    rows = np.where(band_mask(d["lat"], lo, hi) & usable_mask(d, date))[0]
    y = labels_for(d, date, propagate)[rows]
    name = entry if "@" in entry else f"{entry}@{date}"
    return {"name": name, "city": cty, "date": date, "rows": rows, "y": y,
            "xy": d["xy"][rows], "lon": d["lon"][rows], "lat": d["lat"][rows],
            "table": d["table"].iloc[rows]}


def build_parts(entries, propagate=False):
    parts = []
    for entry in entries:
        cty, lo, hi, date = parse_entry(entry)
        if date is None and propagate and len(CITY_REGISTRY[cty]["label_dates"]) > 1:
            # no date pinned and temporal labels on: one part per date
            for dt in CITY_REGISTRY[cty]["label_dates"]:
                parts.append(make_part(f"{entry}@{dt}", propagate=True))
        else:
            parts.append(make_part(entry, propagate=propagate))
    return parts


def part_source(p, offset=0):
    """(PatchSource, positions) for a part at one temporal offset.

    positions says where in the part each returned row sits, so a score
    vector can be scattered back into a full-length array. At offset 0 every
    row is present by construction; at other offsets a few may be missing.
    """
    d = city_data(p["city"])
    if offset == 0:
        date, window = p["date"], None
    else:
        date, window = shift_date(p["date"], offset), temporal_window_days()
    src, rows = city_source(d, date, CONFIG["features"], rows=p["rows"],
                            window_days=window)
    return src, np.searchsorted(p["rows"], rows)


def validate_spatial_bands(split_cfg):
    """Require disjoint, complete geographic development bands."""
    roles = ("train", "stack", "val", "test")
    bands = strip_dates({k: split_cfg.get(k, []) for k in roles})
    cities = sorted({parse_split_entry(entry)[0]
                     for entries in bands.values() for entry in entries})
    for city in cities:
        lat = city_data(city)["lat"]
        masks = {}
        for role in roles:
            mask = np.zeros(len(lat), dtype=bool)
            for entry in bands[role]:
                name, lo, hi = parse_split_entry(entry)
                if name == city:
                    mask |= band_mask(lat, lo, hi)
            masks[role] = mask
        for i, left in enumerate(roles):
            for right in roles[i + 1:]:
                overlap = masks[left] & masks[right]
                if overlap.any():
                    raise RuntimeError(
                        f"{city}: {overlap.sum():,} buildings occur in both "
                        f"{left} and {right} bands.")
        assigned = np.logical_or.reduce([masks[r] for r in roles])
        if not assigned.all():
            raise RuntimeError(
                f"{city}: {(~assigned).sum():,} buildings are outside all "
                "development bands.")


validate_spatial_bands(SPLIT)
train_parts = build_parts(SPLIT["train"], propagate=CONFIG["label_temporal"])
val_parts = build_parts(SPLIT["val"])
stack_parts = build_parts(SPLIT["stack"]) if CONFIG["stacking"]["enabled"] else []

# PatchSource joins the pre and post arrays channel-wise, keeps one latitude
# band and joins dates end to end without reading the pixel arrays. The
# patches stay memory-mapped on disk, so only the batch in use is ever in RAM.
X_tr = PatchSource.concat([part_source(p)[0] for p in train_parts])
y_tr = np.concatenate([p["y"] for p in train_parts])
X_va = PatchSource.concat([part_source(p)[0] for p in val_parts])
y_va = np.concatenate([p["y"] for p in val_parts])

# Normalization statistics come from the training data and nowhere else.
#
# In EVALUATE_ONLY mode they are read back from the checkpoint instead of
# recomputed. The statistics are part of the fitted model, so holdout data
# must be normalized with the values used during training. Recomputing them
# would feed the network inputs on a different scale from the one it learned.
SAVED_RUN_TAGS = list_saved_runs(OUT["models"])

if EVALUATE_ONLY:
    if not SAVED_RUN_TAGS:
        raise RuntimeError(
            f"EVALUATE_ONLY is on but {OUT['models']} holds no completed run. "
            "Train the experiment first, or set EVALUATE_ONLY = False.")
    _ref = load_run(OUT["models"], SAVED_RUN_TAGS[0], device="cpu")
    mu, sd = _ref["mu"], _ref["sd"]
    if _ref["channel_names"] != CHANNEL_NAMES:
        raise RuntimeError(
            f"checkpoint was trained on channels {_ref['channel_names']} but "
            f"CONFIG asks for {CHANNEL_NAMES}.")
    del _ref
    print(f"channel statistics reused from checkpoint "
          f"'{SAVED_RUN_TAGS[0]}' (no training patches read)")
else:
    # channel_stats streams the file in batches and accumulates in float64.
    t0 = time.time()
    mu, sd = X_tr.channel_stats()
    print(f"channel statistics from {len(X_tr):,} training patches "
          f"({time.time() - t0:.0f}s)")
for n, m, s in zip(CHANNEL_NAMES, mu.ravel(), sd.ravel()):
    print(f"   {n:14s} mean {m:8.3f}  sd {s:6.3f}")

print(f"\ntrain n={len(y_tr):6d}  damaged={y_tr.mean()*100:5.2f} percent")
print(f"val   n={len(y_va):6d}  damaged={y_va.mean()*100:5.2f} percent")
test_parts = build_parts(SPLIT["test"])
for kind, parts in [("stack", stack_parts), ("test", test_parts)]:
    for p in parts:
        print(f"{kind:5s} {p['name']:28s} n={len(p['y']):6d}  "
              f"damaged={p['y'].mean()*100:5.2f} percent")

# Evaluation bands must share no building with training at any date.
train_ids = set().union(*[set(p["table"]["system:index"]) for p in train_parts])
for p in val_parts + stack_parts:
    overlap = train_ids & set(p["table"]["system:index"])
    if overlap:
        raise RuntimeError(f"{p['name']}: {len(overlap):,} buildings also appear "
                           f"in training. Check the band boundaries.")
print("\nno building appears in both training and development evaluation")


In [ ]:
# Check temporal-offset patch availability before model training.
missing = []
for p in stack_parts + val_parts:
    for off in OFFSETS:
        if off == 0:
            continue
        try:
            src, pos = part_source(p, off)
            frac = len(pos) / max(len(p["y"]), 1)
            if frac < 0.9:
                print(f"   {p['name']} {off:+d}d: only {frac*100:.0f} percent of "
                      f"buildings have a patch, the rest become NaN features")
        except FileNotFoundError as e:
            missing.append((p["name"], off))
if missing:
    print("Missing temporal-offset patches:")
    for name, off in missing[:10]:
        print(f"   {name} {off:+d} days")
    print("Temporal stacking requires these notebook 1 outputs.")
else:
    print(f"all {len(OFFSETS) - 1} offsets available for every scored part")


### Spatial split

Latitude bands keep neighbouring buildings in the same role. This reduces
leakage from spatial autocorrelation and shared pixels between nearby
32-pixel patches. The map displays locations, roles and counts without reading
test labels or test prevalence.


In [ ]:
def plot_split_map(split_cfg, path=None):
    """Map Gaza buildings by their train/stack/validation/test role.

    Restricted to the develop-role bands (train, stack, val, test); the
    holdout cities are whole-city transfer targets, not latitude bands of
    Gaza, so they do not belong on this map.
    """
    develop_roles = ("train", "stack", "val", "test")
    bands = strip_dates({k: v for k, v in split_cfg.items() if k in develop_roles})
    cities = []
    for part in bands:
        for entry in bands[part]:
            c = parse_split_entry(entry)[0]
            if c not in cities:
                cities.append(c)

    colors = {"train": "#2E7D32", "stack": "#6A1B9A", "val": "#EF6C00",
             "test": "#C62828"}

    data = {}
    for c in cities:
        d = city_data(c)
        data[c] = (d["lat"], d["lon"])

    # a degree of longitude is shorter than a degree of latitude away from the
    # equator, so set the aspect by cos(latitude) to get the real shape
    def extent(c):
        lat, lon = data[c]
        k = np.cos(np.radians(lat.mean()))
        return (np.ptp(lon) * k), np.ptp(lat), 1 / k

    panel_w = 3.4
    tallest = max(h / (w or 1) for w, h, _ in map(extent, cities))
    fig, axes = plt.subplots(
        1, len(cities), squeeze=False,
        figsize=(panel_w * len(cities), min(panel_w * tallest + 1.0, 9.0)))

    rows = []
    for ax, c in zip(axes[0], cities):
        lat, lon = data[c]
        role = split_assignment(lat, c, bands)
        for r in develop_roles:
            m = role == r
            if not m.any():
                continue
            ax.scatter(lon[m], lat[m], s=1.0, alpha=0.4, color=colors[r],
                       label=r, linewidths=0)
            rows.append({"city": c, "split": r, "buildings": int(m.sum())})
            ax.text(0.98, lat[m].mean(),
                    f"{r}\nn={m.sum():,}",
                    transform=ax.get_yaxis_transform(), ha="right", va="center",
                    fontsize=8, color=colors[r], fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white",
                              ec=colors[r], alpha=0.85, lw=0.8))
        for part in bands:
            for entry in bands[part]:
                c_, lo, hi = parse_split_entry(entry)
                if c_ == c and (lo > 0 or hi < 1):
                    for q in (lo, hi):
                        ax.axhline(np.quantile(lat, q), color="black", lw=0.7,
                                   ls="--", alpha=0.55)
        ax.set_title(c, fontsize=11, pad=6)
        ax.set_aspect(extent(c)[2])
        ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle("Development bands by latitude, including test",
                 fontsize=12)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

    summary = pd.DataFrame(rows)
    summary = summary.reset_index(drop=True)
    print(summary.to_string(index=False))
    return summary


split_summary = plot_split_map(SPLIT, os.path.join(OUT["figures"], "split_map.png"))


## 3. CNN training

Every architecture uses the same training protocol: class-weighted binary
cross-entropy, early stopping on validation ROC AUC and restoration
of the best checkpoint. Validation loss is recorded separately to distinguish
changes in ranking from changes in probability calibration.

Full-data baseline and HPO-winner runs write an atomic rolling
`*_training.pt` checkpoint after every epoch. It contains current and best
weights, optimizer state, early-stopping state, history and random-number
state. A compatible interrupted run resumes at the following epoch, while a
compatible completed checkpoint is loaded without retraining.


In [ ]:
class PatchDataset(Dataset):
    """Yields one (patch, label) pair, normalized on the fly."""

    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = (np.asarray(self.X[i]).astype(np.float32) - mu[0]) / sd[0]
        return torch.from_numpy(x), torch.tensor(float(self.y[i]))


def make_loader(X, y, batch_size, shuffle=False, drop_last=False):
    return DataLoader(PatchDataset(X, y), batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=CONFIG["num_workers"],
                      pin_memory=(device == "cuda"),
                      persistent_workers=CONFIG["num_workers"] > 0)


def atomic_torch_save(payload, path):
    """Write a checkpoint completely before replacing the previous one."""
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)


def train(params, tag, X_train=None, y_train=None, max_epochs=None,
          patience=None, trial=None, save=True, verbose=True, log_every=None):
    """Train one CNN, resuming saved full-data runs after interruptions."""
    X_train = X_tr if X_train is None else X_train
    y_train = y_tr if y_train is None else y_train
    max_epochs = max_epochs or CONFIG["max_epochs"]
    patience = patience or CONFIG["early_stopping_patience"]
    params = dict(params)

    final_path = os.path.join(OUT["models"], f"{tag}.pt") if save else None
    resume_path = (os.path.join(OUT["models"], f"{tag}_training.pt")
                   if save else None)

    def matches(state):
        return (state.get("model") == CONFIG["model"] and
                state.get("channel_names") == CHANNEL_NAMES and
                state.get("params") == params and
                state.get("max_epochs") == max_epochs and
                state.get("patience") == patience and
                state.get("train_rows") == len(y_train) and
                state.get("selection_metric") == CONFIG["selection_metric"])

    torch.manual_seed(CONFIG["seed"])
    model = build_model(CONFIG["model"], CHANNEL_NAMES, device=device,
                        quiet=not verbose, **params)

    # A completed matching run is immutable: reuse it instead of retraining.
    if final_path and os.path.exists(final_path):
        state = torch.load(final_path, map_location="cpu", weights_only=False)
        if state.get("complete") and matches(state):
            model.load_state_dict(state["state_dict"])
            model.to(device)
            if verbose:
                print(f"reused completed run {tag}: epoch {state['best_epoch']}, "
                      f"validation ROC AUC {state['val_roc_auc']:.4f}")
            return {"tag": tag, "model": model, "params": params,
                    "history": np.asarray(state["history"], float),
                    "best_epoch": int(state["best_epoch"]),
                    "val_roc_auc": float(state["val_roc_auc"])}

    batch_size = params.get("batch_size", CONFIG["batch_size"])
    tr_dl = make_loader(X_train, y_train, batch_size, shuffle=True,
                        drop_last=len(y_train) >= 2 * batch_size)
    va_dl = make_loader(X_va, y_va, 512)

    pos_weight = torch.tensor([(len(y_train) - y_train.sum()) /
                               max(y_train.sum(), 1)],
                              dtype=torch.float32, device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(),
                            lr=params.get("learning_rate", CONFIG["learning_rate"]),
                            weight_decay=params.get("weight_decay",
                                                    CONFIG["weight_decay"]))

    @torch.no_grad()
    def evaluate():
        model.eval()
        probs, loss_sum, seen = [], 0.0, 0
        for xb, yb in va_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device)
            logits = model(xb)
            loss_sum += crit(logits, yb).item() * len(yb)
            seen += len(yb)
            probs.append(torch.sigmoid(logits).cpu())
        p = torch.cat(probs).numpy()
        return loss_sum / seen, roc_auc_score(y_va, p)

    start_epoch = 1
    best_auc, best_epoch, best_state, bad, history = -np.inf, -1, None, 0, []
    if resume_path and os.path.exists(resume_path):
        state = torch.load(resume_path, map_location="cpu", weights_only=False)
        if matches(state):
            model.load_state_dict(state["state_dict"])
            model.to(device)
            opt.load_state_dict(state["optimizer_state_dict"])
            start_epoch = int(state["completed_epoch"]) + 1
            best_auc = float(state["best_auc"])
            best_epoch = int(state["best_epoch"])
            best_state = state["best_state"]
            bad = int(state["bad_epochs"])
            history = [tuple(row) for row in state["history"]]
            if "torch_rng_state" in state:
                torch.set_rng_state(state["torch_rng_state"])
            if device == "cuda" and state.get("cuda_rng_state_all") is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])
            if verbose:
                print(f"resuming {tag} at epoch {start_epoch} "
                      f"(best epoch {best_epoch}, ROC AUC {best_auc:.4f})")
        elif verbose:
            print(f"ignored incompatible checkpoint {resume_path}")

    # If the previous epoch already triggered early stopping, only the
    # final atomic save remains; do not train one extra epoch on resume.
    epoch_range = range(start_epoch, max_epochs + 1) if bad < patience else ()
    for ep in epoch_range:
        model.train()
        tot, seen = 0.0, 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(yb)
            seen += len(yb)
        train_loss = tot / max(seen, 1)
        val_loss, val_auc = evaluate()
        history.append((ep, train_loss, val_loss, val_auc))

        star = ""
        if best_state is None or val_auc > best_auc + 1e-4:
            best_auc, best_epoch, bad = val_auc, ep, 0
            best_state = copy.deepcopy(model.state_dict())
            star = "  best so far"
        else:
            bad += 1
        periodic_log = (log_every is not None and log_every > 0 and
                        (ep == start_epoch or ep % log_every == 0 or
                         ep == max_epochs or bad >= patience))
        if verbose or periodic_log:
            print(f"{tag}  epoch {ep:3d}/{max_epochs}  train loss {train_loss:.4f}  "
                  f"val loss {val_loss:.4f}  val ROC AUC {val_auc:.4f}{star}")

        if resume_path:
            atomic_torch_save({
                "complete": False, "completed_epoch": ep,
                "state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "best_state": best_state, "best_auc": float(best_auc),
                "best_epoch": best_epoch, "bad_epochs": bad,
                "history": history, "params": params,
                "model": CONFIG["model"],
                "channel_names": CHANNEL_NAMES,
                "max_epochs": max_epochs, "patience": patience,
                "train_rows": len(y_train),
                "selection_metric": CONFIG["selection_metric"],
                "torch_rng_state": torch.get_rng_state(),
                "cuda_rng_state_all": (torch.cuda.get_rng_state_all()
                                           if device == "cuda" else None),
            }, resume_path)

        if trial is not None:
            trial.report(val_auc, ep)
            if trial.should_prune():
                if not periodic_log:
                    print(f"{tag}  pruned at epoch {ep}/{max_epochs}  "
                          f"val ROC AUC {val_auc:.4f}")
                raise optuna.TrialPruned()
        if bad >= patience:
            if verbose:
                print(f"early stop at epoch {ep}, best was epoch {best_epoch}")
            break

    model.load_state_dict(best_state)
    model.to(device)
    history_array = np.asarray(history, float)
    run = {"tag": tag, "model": model, "params": params,
           "history": history_array, "best_epoch": best_epoch,
           "val_roc_auc": float(best_auc)}

    if final_path:
        atomic_torch_save({
            "complete": True, "state_dict": best_state,
            "mu": mu, "sd": sd, "history": history_array,
            "channel_names": CHANNEL_NAMES, "params": params,
            "model": CONFIG["model"], "config": CONFIG,
            "best_epoch": best_epoch, "val_roc_auc": float(best_auc),
            "max_epochs": max_epochs, "patience": patience,
            "train_rows": len(y_train),
                "selection_metric": CONFIG["selection_metric"],
        }, final_path)
        if verbose:
            print(f"saved completed run {final_path}  (best epoch {best_epoch}, "
                  f"validation ROC AUC {best_auc:.4f})")
    return run


RUNS = {}
base_params = dict(CONFIG["model_params"],
                   learning_rate=CONFIG["learning_rate"],
                   weight_decay=CONFIG["weight_decay"],
                   batch_size=CONFIG["batch_size"])

In [ ]:
# Show the configured baseline architecture before fitting any weights.
model_to_summarize = build_model(
    CONFIG["model"], CHANNEL_NAMES, device=device, quiet=True, **base_params)
sample_x = np.asarray(X_tr[0], dtype=np.float32)
input_shape = (1, *sample_x.shape)  # batch, channels, height, width

summary(
    model_to_summarize,
    input_size=input_shape,
    device=device,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    depth=5,
)
del model_to_summarize

In [ ]:
if EVALUATE_ONLY:
    # Load every completed run this experiment saved. load_run() rebuilds the
    # architecture from the checkpoint itself, so nothing here depends on the
    # training data still being reachable.
    for tag in SAVED_RUN_TAGS:
        RUNS[tag] = load_run(OUT["models"], tag, device=device)
        r = RUNS[tag]
        print(f"loaded {tag}: epoch {r['best_epoch']}, "
              f"validation ROC AUC {r['val_roc_auc']:.4f}, params {r['params']}")
    print(f"\nEVALUATE_ONLY: {len(RUNS)} run(s) loaded, none trained.")
else:
    RUNS["baseline"] = train(base_params, "baseline")


## 4. Hyperparameter search with Optuna

Optuna uses a TPE sampler, median pruning and validation ROC AUC as the
objective. Trials train on a fixed subsample with a shorter epoch budget;
the winning parameter set is retrained on the complete training band.

The study is stored in `optuna.db` with `load_if_exists=True`. A callback
atomically refreshes the trial CSV, best-parameter JSON and sampler state after
each completed or pruned trial. `hpo.n_trials` is the target total, so raising
it before final selection adds only the missing trials. Other scientific
configuration changes require a new experiment name.

All trials use the same spatial validation bands. Validation results therefore
describe performance for this predefined split rather than variation across
alternative geographic partitions.


In [ ]:
def hpo_search_space(trial):
    """Edit here to change what is searched."""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "width": trial.suggest_categorical("width", [8, 16, 32]),
        "depth": trial.suggest_int("depth", 2, 4),
    }
    if CONFIG["model"] == "siamese":
        params["head_dim"] = trial.suggest_categorical("head_dim", [64, 128, 256])
    return params


def atomic_pickle_dump(value, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as fh:
        pickle.dump(value, fh)
    os.replace(tmp, path)


def atomic_json_dump(value, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(value, fh, indent=2)
    os.replace(tmp, path)


def atomic_csv_dump(frame, path, **kwargs):
    tmp = path + ".tmp"
    frame.to_csv(tmp, **kwargs)
    os.replace(tmp, path)


def persist_hpo_state(study, _trial=None):
    """Export a human-readable snapshot after every finished trial."""
    trials_path = os.path.join(OUT["metrics"], "hpo_trials.csv")
    frame = study.trials_dataframe(
        attrs=("number", "value", "params", "state",
               "datetime_start", "datetime_complete"))
    atomic_csv_dump(frame, trials_path, index=False)

    complete = [t for t in study.trials if t.state.name == "COMPLETE"]
    if complete:
        best = study.best_trial
        atomic_json_dump({"best_trial": best.number,
                          "best_value": best.value,
                          "selection_metric": "validation_roc_auc",
                          "best_params": best.params},
                         os.path.join(OUT["metrics"], "hpo_best_params.json"))

    # SQLite persists trials, but not the TPE sampler's internal state.
    atomic_pickle_dump(study.sampler, os.path.join(EXP_ROOT, "optuna_sampler.pkl"))


def run_hpo(cfg):
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    h = cfg["hpo"]

    # a fixed subsample, so every trial is scored on the same data
    rng = np.random.default_rng(cfg["seed"])
    n = len(y_tr)
    keep = np.sort(rng.choice(n, size=int(h["train_subsample"] * n),
                              replace=False)) if h["train_subsample"] < 1 else np.arange(n)
    X_sub, y_sub = X_tr.rows(keep), y_tr[keep]
    print(f"trials train on {len(y_sub):,} of {n:,} patches, "
          f"up to {h['max_epochs']} epochs each")

    def objective(trial):
        params = hpo_search_space(trial)
        run = train(params, f"trial{trial.number}", X_train=X_sub, y_train=y_sub,
                    max_epochs=h["max_epochs"], patience=h["patience"],
                    trial=trial, save=False, verbose=False,
                    log_every=HPO_LOG_EVERY)
        print(f"  trial {trial.number:3d}  validation ROC AUC {run['val_roc_auc']:.4f}  "
              + "  ".join(f"{k}={v:g}" if isinstance(v, float) else f"{k}={v}"
                          for k, v in params.items()))
        return run["val_roc_auc"]

    sampler_path = os.path.join(EXP_ROOT, "optuna_sampler.pkl")
    sampler = optuna.samplers.TPESampler(seed=cfg["seed"])
    if os.path.exists(sampler_path):
        try:
            with open(sampler_path, "rb") as fh:
                sampler = pickle.load(fh)
            print("restored Optuna sampler state")
        except Exception as exc:
            warnings.warn(f"could not restore Optuna sampler; using a fresh one: {exc}")

    study = optuna.create_study(
        direction="maximize",
        study_name=cfg["experiment_name"],
        storage=f"sqlite:///{os.path.join(EXP_ROOT, 'optuna.db')}",
        load_if_exists=True,
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
    done = len([t for t in study.trials
                if t.state.name in ("COMPLETE", "PRUNED")])
    remaining = max(h["n_trials"] - done, 0)
    selection_locked = os.path.exists(
        os.path.join(OUT["metrics"], "final_selection.json"))
    if remaining and selection_locked:
        print("Final selection is frozen; no additional HPO trials will run.")
        remaining = 0
    if remaining:
        study.optimize(objective, n_trials=remaining,
                       timeout=(h["timeout_minutes"] or 0) * 60 or None,
                       callbacks=[persist_hpo_state])
    else:
        print(f"study already has {done} trials, reusing it")
    persist_hpo_state(study)
    return study


if EVALUATE_ONLY:
    # The HPO winner, if there was one, is already among the loaded runs and
    # its cache_key must match the one used when its scores were cached.
    study = None
    # The cached CNN scores of the HPO run are keyed by its trial number, so
    # restore that key or score_all() would recompute them under a new name.
    hpo_state = os.path.join(OUT["metrics"], "hpo_best_params.json")
    if "hpo" in RUNS and os.path.exists(hpo_state):
        try:
            with open(hpo_state) as fh:
                best_number = json.load(fh)["best_trial"]
            RUNS["hpo"]["cache_key"] = f"hpo_best_trial_{best_number}"
            print(f"restored HPO cache key: hpo_best_trial_{best_number}")
        except Exception as exc:
            warnings.warn(f"could not restore the HPO cache key: {exc}")
    print("EVALUATE_ONLY: HPO skipped, "
          f"loaded runs = {list(RUNS)}")
elif CONFIG["hpo"]["enabled"]:
    study = run_hpo(CONFIG)
    print(f"\nbest trial {study.best_trial.number}: "
          f"validation ROC AUC {study.best_value:.4f}")
    print("best parameters:")
    for k, v in study.best_params.items():
        print(f"   {k}: {v}")

    hpo_params = dict(base_params)
    hpo_params.update(study.best_params)
    hpo_cache_key = f"hpo_best_trial_{study.best_trial.number}"
    print(f"\nretraining HPO winner trial {study.best_trial.number} "
          "on the full training set:")
    hpo_run = train(hpo_params, "hpo")
    hpo_run["cache_key"] = hpo_cache_key
    RUNS["hpo"] = hpo_run
else:
    study = None
    print("HPO disabled")

print("\nruns to evaluate:", {k: round(v["val_roc_auc"], 4) for k, v in RUNS.items()})

In [ ]:
# Summarize the completed search and show the sampled parameter ranges.
if study is not None:
    complete = [t for t in study.trials
                if t.state == optuna.trial.TrialState.COMPLETE
                and t.value is not None]
    if len(complete) >= 3:
        hist = pd.DataFrame([dict(number=t.number, value=t.value, **t.params)
                             for t in complete])
        print(hist.sort_values("value", ascending=False)
              .head(8).round(5).to_string(index=False))
        ranges = []
        for name in sorted(set().union(*(t.params for t in complete))):
            values = hist[name].dropna()
            numeric = pd.to_numeric(values, errors="coerce")
            ranges.append({
                "parameter": name,
                "n_unique": int(values.nunique()),
                "minimum": float(numeric.min()) if numeric.notna().all() else None,
                "maximum": float(numeric.max()) if numeric.notna().all() else None,
                "values": sorted(values.unique().tolist()),
            })
        print("\nExplored parameter values:")
        print(pd.DataFrame(ranges).to_string(index=False))
        try:
            importance = optuna.importance.get_param_importances(study)
            print("\nParameter importance for validation ROC AUC:")
            for name, value in importance.items():
                print(f"   {name:16s} {value:.3f}")
        except Exception as exc:
            print(f"Parameter importance unavailable: {exc}")

        figures = {
            "hpo_optimization_history.html":
                optuna.visualization.plot_optimization_history(study),
            "hpo_param_importances.html":
                optuna.visualization.plot_param_importances(study),
            "hpo_slice.html": optuna.visualization.plot_slice(study),
        }
        for filename, figure in figures.items():
            figure.write_html(os.path.join(OUT["figures"], filename))
            figure.show()
    else:
        print("At least three completed trials are required for HPO plots.")

## 5. Scoring temporal offsets

Each trained CNN scores every development part at the configured temporal
offsets. Only the post-event imagery changes; the pre-event composite remains
fixed. `part_source` creates a lazy memory-mapped view, and
`predict_probs` loads one float32 batch at a time.

Missing offset patches produce NaN scores while preserving row alignment.
Completed score arrays are cached atomically under `predictions/`. Test
predictions use a separate cache created only after model selection.


In [ ]:
def score_part(run, p, offsets=(0,)):
    """{offset: score array} for one part, NaN where the patch is missing."""
    out = {}
    for off in offsets:
        s = np.full(len(p["y"]), np.nan, np.float32)
        try:
            src, pos = part_source(p, off)
            s[pos] = predict_probs(run["model"], src, mu, sd, device=device,
                                   batch=CONFIG["predict_batch"])
        except FileNotFoundError:
            warnings.warn(f"{p['name']}: no patches at offset {off:+d}, "
                          f"temporal features there will be NaN")
        out[off] = s
    return out


def score_cache_path(tag, part_name, cache_group):
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in part_name)
    group = "".join(c if c.isalnum() or c in "-_." else "_" for c in cache_group)
    return os.path.join(OUT["predictions"], f"{group}_cnn_scores_{tag}_{safe}.npz")


def score_all(run, parts, offsets, cache_group="development"):
    """Score every part, saving each completed part for crash-safe reuse."""
    t0 = time.time()
    scores = {}
    expected_offsets = np.asarray(offsets, dtype=np.int32)
    cache_key = run.get("cache_key", run["tag"])
    for p in parts:
        path = score_cache_path(cache_key, p["name"], cache_group)
        cached = None
        if os.path.exists(path):
            try:
                with np.load(path) as data:
                    saved_offsets = data["offsets"]
                    saved_scores = data["scores"]
                if (np.array_equal(saved_offsets, expected_offsets) and
                        saved_scores.shape == (len(offsets), len(p["y"]))):
                    cached = {off: saved_scores[i] for i, off in enumerate(offsets)}
            except Exception as exc:
                warnings.warn(f"ignoring unreadable score cache {path}: {exc}")

        if cached is not None:
            scores[p["name"]] = cached
            print(f"  loaded cached scores: {run['tag']} / {p['name']}")
            continue

        part_scores = score_part(run, p, offsets)
        matrix = np.stack([part_scores[off] for off in offsets])
        tmp = path + ".tmp.npz"
        np.savez_compressed(tmp, offsets=expected_offsets, scores=matrix)
        os.replace(tmp, path)
        scores[p["name"]] = part_scores

    n = sum(len(p["y"]) for p in parts)
    print(f"  {run['tag']}: {n:,} buildings x {len(offsets)} offsets "
          f"in {time.time() - t0:.0f}s")
    return scores


SCORED = stack_parts + val_parts
SCORES = {}
for tag, run in RUNS.items():
    SCORES[tag] = score_all(run, SCORED, OFFSETS)

# Summarize the temporal score sequence as a data-quality check.
p = val_parts[0]
s = SCORES[list(RUNS)[0]][p["name"]]
frame = pd.DataFrame({f"{o:+d}d": [np.nanmean(s[o][p["y"] == 0]),
                                   np.nanmean(s[o][p["y"] == 1])]
                      for o in OFFSETS}, index=["intact", "damaged"])
print(f"\nmean CNN score by offset, {p['name']}:")
print(frame.round(4).to_string())
print(f"missing scores per offset: "
      f"{ {o: int(np.isnan(s[o]).sum()) for o in OFFSETS} }")


## 6. Spatial and temporal second stage

The first-stage CNN classifies each image patch independently. XGBoost then
uses a deliberately compact set of CNN-score features. It never receives raw
SAR pixels, coordinates, labels from neighbouring buildings or assessment
metadata. Neighbours are calculated separately within each split part and
assessment date.

The spatial-only stacker uses the first five features below. The
spatial-temporal stacker uses all eleven in this exact order:

| feature | meaning |
|---|---|
| `score_t0` | CNN damage probability at the labelled assessment date |
| `k8_mean` | mean `score_t0` of the eight nearest buildings |
| `k8_std` | standard deviation of those eight scores |
| `k32_mean` | mean `score_t0` of the 32 nearest buildings |
| `k32_std` | standard deviation of those 32 scores |
| `score_t-24` | CNN probability from post-event imagery 24 days before the assessment |
| `score_t-12` | CNN probability 12 days before the assessment |
| `score_t+12` | CNN probability 12 days after the assessment |
| `score_t+24` | CNN probability 24 days after the assessment |
| `pre_change` | `score_t0 - mean(score_t-24, score_t-12)` |
| `post_persistence` | `mean(score_t+12, score_t+24) - score_t0` |

XGBoost handles missing offset scores natively. `pre_change` and
`post_persistence` use the available member when one of their two supporting
scores is missing and remain missing when neither is available. Because the
last four temporal predictors include future imagery, this is a retrospective
damage classifier rather than a real-time assessment-date model.

Each stacker is fitted on the dedicated stack band and uses validation for
early stopping.


In [ ]:
def feature_table(parts, scores, ks, temporal):
    """Second-stage features for a set of parts, one region+date at a time."""
    tables, labels = [], []
    for p in parts:
        s = scores[p["name"]]
        f = neighbour_features(p["xy"], np.nan_to_num(s[0], nan=0.0), ks=tuple(ks))
        if temporal:
            f = pd.concat([f, temporal_features(s)], axis=1)
        tables.append(f)
        labels.append(p["y"])
    return pd.concat(tables, ignore_index=True), np.concatenate(labels)


def train_stacker(cfg, scores, temporal, tag):
    sc = cfg["stacking"]
    X_stack, y_stack = feature_table(stack_parts, scores, sc["ks"], temporal)
    X_val, y_val = feature_table(val_parts, scores, sc["ks"], temporal)
    expected_columns = list(sc["spatial_features"])
    if temporal:
        expected_columns += list(sc["temporal_features"])
    if (list(X_stack.columns) != expected_columns or
            list(X_val.columns) != expected_columns):
        raise RuntimeError(
            f"XGBoost feature schema mismatch: expected {expected_columns}, "
            f"got {list(X_stack.columns)}")

    clf = XGBClassifier(
        n_estimators=sc["n_estimators"], max_depth=sc["max_depth"],
        learning_rate=sc["learning_rate"], subsample=0.8, colsample_bytree=0.8,
        eval_metric="auc", early_stopping_rounds=sc["early_stopping_rounds"],
        scale_pos_weight=(y_stack == 0).sum() / max((y_stack == 1).sum(), 1),
        random_state=cfg["seed"], n_jobs=4)
    clf.fit(X_stack, y_stack, eval_set=[(X_val, y_val)], verbose=False)

    val_p = clf.predict_proba(X_val)[:, 1]
    val_roc_auc = roc_auc_score(y_val, val_p)
    imp = (pd.Series(clf.feature_importances_, index=X_stack.columns)
             .sort_values(ascending=False))
    print(f"  {tag}: {len(X_stack):,} rows x {X_stack.shape[1]} features, "
          f"best iteration {clf.best_iteration}, validation ROC AUC {val_roc_auc:.4f}")
    return {"model": clf, "val_roc_auc": float(val_roc_auc),
            "importance": imp, "temporal": temporal,
            "columns": list(X_stack.columns)}


stacker_path = os.path.join(OUT["models"], "stackers.pt")
EXPECTED_RUN_KEYS = {tag: run.get("cache_key", tag)
                     for tag, run in RUNS.items()}
STACKER_RUN_KEYS = {}
STACKER_NEEDS_MIGRATION = False
if os.path.exists(stacker_path):
    try:
        saved_stackers = torch.load(stacker_path, map_location="cpu", weights_only=False)
        saved_cfg = saved_stackers.get("config")
        saved_format = saved_stackers.get("format")
        STACKER_NEEDS_MIGRATION = saved_format == 1
        if (saved_format not in (1, 2) or saved_cfg is None or
                experiment_identity(saved_cfg) != experiment_identity(CONFIG)):
            raise ValueError("legacy or incompatible stacker checkpoint")
        STACKERS = saved_stackers["stackers"]
        STACKER_RUN_KEYS = dict(saved_stackers.get("run_keys", {}))
        if saved_format == 1:
            # Baseline identity was unambiguous in the earlier format. An
            # HPO stacker is trusted only after selection has been frozen;
            # otherwise an expanded study may have produced a new winner.
            if "baseline" in STACKERS:
                STACKER_RUN_KEYS["baseline"] = EXPECTED_RUN_KEYS["baseline"]
            if ("hpo" in STACKERS and
                    os.path.exists(os.path.join(OUT["metrics"],
                                                "final_selection.json"))):
                STACKER_RUN_KEYS["hpo"] = EXPECTED_RUN_KEYS["hpo"]
        for tag in list(STACKERS):
            if (tag not in EXPECTED_RUN_KEYS or
                    STACKER_RUN_KEYS.get(tag) != EXPECTED_RUN_KEYS[tag]):
                print(f"discarding stale stackers for {tag}")
                STACKERS.pop(tag, None)
                STACKER_RUN_KEYS.pop(tag, None)
        print(f"loaded saved stackers: {list(STACKERS)}")
    except Exception as exc:
        warnings.warn(f"could not safely reuse saved stackers; retraining them: {exc}")
        STACKERS = {}
else:
    STACKERS = {}


def save_stackers():
    atomic_torch_save({"format": 2, "config": CONFIG,
                       "run_keys": STACKER_RUN_KEYS,
                       "stackers": STACKERS}, stacker_path)


if STACKER_NEEDS_MIGRATION and STACKERS:
    save_stackers()
    print("migrated stacker metadata to format 2")


if EVALUATE_ONLY:
    # Nothing may be fitted here: a stacker refitted now would be a different
    # model from the one final_selection.json was frozen against.
    required_variants = ["xgb_spatial"]
    if CONFIG["temporal_stacking"]["enabled"]:
        required_variants.append("xgb_spatial_temporal")
    missing = [f"{tag}/{variant}" for tag in RUNS
               for variant in required_variants
               if variant not in STACKERS.get(tag, {})]
    if CONFIG["stacking"]["enabled"] and missing:
        raise RuntimeError(
            f"EVALUATE_ONLY is on but {stacker_path} has no stacker for "
            f"{missing}. A training-mode run is required to fit them.")
    print(f"EVALUATE_ONLY: reusing saved stackers for {list(STACKERS)}, "
          "none fitted.")
elif CONFIG["stacking"]["enabled"]:
    for tag in RUNS:
        STACKERS.setdefault(tag, {})
        STACKER_RUN_KEYS[tag] = EXPECTED_RUN_KEYS[tag]
        print(f"{tag}:")
        if "xgb_spatial" not in STACKERS[tag]:
            STACKERS[tag]["xgb_spatial"] = train_stacker(
                CONFIG, SCORES[tag], temporal=False, tag="spatial only")
            save_stackers()
        else:
            print("  reusing saved spatial-only stacker")

        if CONFIG["temporal_stacking"]["enabled"]:
            if "xgb_spatial_temporal" not in STACKERS[tag]:
                STACKERS[tag]["xgb_spatial_temporal"] = train_stacker(
                    CONFIG, SCORES[tag], temporal=True, tag="spatial + temporal")
                save_stackers()
            else:
                print("  reusing saved spatial+temporal stacker")

In [ ]:
# Compare aggregate and individual feature importance for the temporal and
# spatial feature blocks in the fitted second-stage model.
tag = "hpo" if "hpo" in STACKERS else "baseline"
if STACKERS and "xgb_spatial_temporal" in STACKERS[tag]:
    imp = STACKERS[tag]["xgb_spatial_temporal"]["importance"]
    temporal_columns = set(CONFIG["stacking"]["temporal_features"])
    kind = pd.Series(
        ["temporal" if name in temporal_columns else "spatial"
         for name in imp.index], index=imp.index)
    print(f"total importance: {kind.groupby(kind).apply(lambda s: imp[s.index].sum()).round(3).to_dict()}")

    top = imp.head(20)[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top.index, top.values,
            color=["C1" if kind[k] == "temporal" else "C0" for k in top.index])
    ax.set_xlabel("XGBoost gain importance")
    ax.set_title(f"{tag}: what the second stage uses\n"
                 f"(orange = temporal, blue = spatial)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT["figures"], "stacker_importance.png"), dpi=150)
    plt.show()


## 7. Validation model selection

Every candidate `(CNN run, prediction variant)` is compared here using only
validation data. The selection metric is mean ROC AUC across the validation dates, giving each assessment date equal weight.

ROC AUC is threshold-free, so it is used to choose the candidate.
After selection, one
F1-maximising threshold is fitted to that candidate's pooled validation
predictions. The selected run, variant, metric, and threshold are atomically
written to `metrics/final_selection.json` before test data is constructed.

Validation is also used for HPO and early stopping, so its scores are
development estimates rather than final performance. The sealed test result
at the end of the notebook is the only final estimate.


In [ ]:
def pr_auc_score(y_true, score):
    """Trapezoidal area under the precision-recall curve."""
    precision, recall, _ = precision_recall_curve(y_true, score)
    return auc(recall, precision)


def binary_metrics(y_true, score, thresh):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    valid = np.isfinite(score)
    y_true, score = y_true[valid], score[valid]
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, (score >= thresh).astype(int), average="binary", zero_division=0)
    two_classes = np.unique(y_true).size == 2
    return {"P": p, "R": r, "F1": f1,
            "PR_AUC": (pr_auc_score(y_true, score)
                       if two_classes else np.nan),
            "ROC_AUC": (roc_auc_score(y_true, score)
                        if two_classes else np.nan),
            "prevalence": float(np.mean(y_true)), "n": len(y_true)}


def variant_scores(tag, p, variant, score_store=None):
    """Scores for one variant on one part from the supplied CNN scores."""
    score_store = SCORES if score_store is None else score_store
    s0 = np.nan_to_num(score_store[tag][p["name"]][0], nan=0.0)
    if variant == "cnn":
        return s0
    if variant == "cnn+fixed_neighbor_rule":
        sm = CONFIG["spatial_smoothing"]
        return spatial_smooth(p["xy"], s0, k=sm["k"], weight=sm["weight"])
    st = STACKERS[tag][variant]
    feats, _ = feature_table([p], score_store[tag], CONFIG["stacking"]["ks"],
                             st["temporal"])
    return st["model"].predict_proba(feats[st["columns"]])[:, 1]


VARIANTS = ["cnn", "cnn+fixed_neighbor_rule"]
if CONFIG["stacking"]["enabled"]:
    VARIANTS.append("xgb_spatial")
    if CONFIG["temporal_stacking"]["enabled"]:
        VARIANTS.append("xgb_spatial_temporal")

# Compare every candidate on validation. Selection uses equal weight per
# validation part/date; its threshold is fitted to pooled validation rows.
validation_rows, selection_rows, THRESHOLDS = {}, [], {}
for tag in RUNS:
    THRESHOLDS[tag] = {}
    for variant in VARIANTS:
        part_scores = [variant_scores(tag, p, variant) for p in val_parts]
        pooled_y = np.concatenate([p["y"] for p in val_parts])
        pooled_s = np.concatenate(part_scores)
        threshold, _ = best_f1_threshold(pooled_y, pooled_s)
        THRESHOLDS[tag][variant] = float(threshold)

        part_auc = []
        for p, score in zip(val_parts, part_scores):
            p.setdefault("scores", {})[(tag, variant)] = score
            metrics = binary_metrics(p["y"], score, threshold)
            validation_rows[(tag, variant, p["name"])] = metrics
            part_auc.append(metrics["ROC_AUC"])
        selection_rows.append({"run": tag, "variant": variant,
                               "mean_val_ROC_AUC": float(np.mean(part_auc)),
                               "pooled_val_ROC_AUC": float(
                                   roc_auc_score(pooled_y, pooled_s)),
                               "val_threshold": float(threshold)})

VAL_RESULTS = pd.DataFrame(validation_rows).T
VAL_RESULTS.index.names = ["run", "variant", "region"]
atomic_csv_dump(
    VAL_RESULTS.reset_index(),
    os.path.join(OUT["metrics"], "validation_metrics_by_part.csv"),
    index=False)
MODEL_SELECTION = (pd.DataFrame(selection_rows)
                   .sort_values(["mean_val_ROC_AUC", "run", "variant"],
                                ascending=[False, True, True])
                   .reset_index(drop=True))
atomic_csv_dump(MODEL_SELECTION,
                os.path.join(OUT["metrics"], "validation_model_selection.csv"),
                index=False)
print("validation candidate ranking (selection metric: mean_val_ROC_AUC):\n")
print(MODEL_SELECTION.round(4).to_string(index=False))

winner = MODEL_SELECTION.iloc[0]
computed_selection = {
    "experiment_name": CONFIG["experiment_name"],
    "run": str(winner["run"]),
    "variant": str(winner["variant"]),
    "selection_metric": "mean_validation_ROC_AUC",
    "selection_value": float(winner["mean_val_ROC_AUC"]),
    "threshold_source": "pooled_validation_best_F1",
    "threshold": float(winner["val_threshold"]),
}
selection_path = os.path.join(OUT["metrics"], "final_selection.json")
if os.path.exists(selection_path):
    with open(selection_path) as fh:
        FINAL_SELECTION = json.load(fh)
    same_choice = all(FINAL_SELECTION.get(k) == computed_selection[k]
                      for k in ("experiment_name", "run", "variant",
                                "selection_metric", "threshold_source"))
    same_threshold = np.isclose(FINAL_SELECTION.get("threshold", np.nan),
                                computed_selection["threshold"],
                                rtol=0, atol=1e-10)
    if not (same_choice and same_threshold):
        raise RuntimeError(
            "The frozen final selection differs from the current validation "
            "winner. Use a new experiment_name; do not silently change a "
            "pipeline after its test evaluation.")
    print(f"\nreusing frozen final selection from {selection_path}")
else:
    FINAL_SELECTION = computed_selection
    atomic_json_dump(FINAL_SELECTION, selection_path)
    print(f"\nfroze final selection in {selection_path}")

FINAL_TAG = FINAL_SELECTION["run"]
FINAL_VARIANT = FINAL_SELECTION["variant"]
FINAL_THRESHOLD = float(FINAL_SELECTION["threshold"])
print(f"Final pipeline: {FINAL_TAG} / {FINAL_VARIANT}, "
      f"validation threshold={FINAL_THRESHOLD:.4f}")


In [ ]:
# Validation ROC AUC by candidate and assessment date.
head = (VAL_RESULTS.reset_index()
        .pivot_table(index=["variant", "run"], columns="region", values="ROC_AUC"))
order = [(v, t) for v in VARIANTS for t in RUNS]
head = head.reindex([i for i in order if i in head.index]).round(4)
print("ROC AUC (higher is better)\n")
print(head.to_string())
atomic_csv_dump(head, os.path.join(OUT["metrics"], "validation_roc_auc_by_part.csv"))

# The predeclared selection metric, sorted with the winner at the top.
plot_selection = MODEL_SELECTION.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, max(3.5, 0.42 * len(plot_selection))))
labels = [f"{r} / {v}" for r, v in
          zip(plot_selection["run"], plot_selection["variant"])]
ax.barh(labels, plot_selection["mean_val_ROC_AUC"], color="C0")
ax.set_xlabel("mean validation ROC AUC")
ax.set_title("Candidate selection on validation only")
plt.tight_layout()
plt.savefig(os.path.join(OUT["figures"], "validation_variant_comparison.png"), dpi=150,
            bbox_inches="tight")
plt.show()


In [ ]:
# Mean ROC AUC by stage. Each delta is relative to the preceding
# prediction variant in this ordered comparison:
#   CNN -> fixed neighbour rule -> learned spatial -> learned spatial+temporal
flat = VAL_RESULTS.reset_index()
metric_columns = ["P", "R", "F1", "PR_AUC", "ROC_AUC"]
validation_comparison = (
    flat.groupby(["run", "variant"])[metric_columns].mean()
        .sort_values("ROC_AUC", ascending=False))
print("Validation metrics averaged equally across assessment dates:\n")
print(validation_comparison.round(4).to_string())
atomic_csv_dump(
    validation_comparison.reset_index(),
    os.path.join(OUT["metrics"], "validation_model_comparison.csv"),
    index=False)

avg = flat.groupby(["run", "variant"])["ROC_AUC"].mean().unstack("variant")
ladder = [v for v in VARIANTS if v in avg.columns]
print("Mean ROC AUC across validation dates:\n")
print(avg[ladder].round(4).to_string())

deltas = avg[ladder].diff(axis=1).round(4)
deltas.columns = [f"+{c}" for c in deltas.columns]
print("\ngain over the previous stage:\n")
print(deltas.iloc[:, 1:].to_string())

if "hpo" in avg.index:
    gain = (avg.loc["hpo"] - avg.loc["baseline"]).round(4)
    print("\nHPO minus baseline, per variant:")
    print(gain[ladder].to_string())
    print("\nHPO minus baseline is reported separately from the stage gains.")


## 8. Validation diagnostics (test remains sealed)


In [ ]:
def plot_training(runs, path=None):
    """Loss curves and validation ROC AUC on shared axes.

    Colors distinguish training from validation:
      blue   = training
      orange = validation

    Line styles distinguish runs:
      solid  = baseline
      dashed = HPO
    """
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    for tag, run in runs.items():
        h = run["history"]
        run_ls = "--" if tag.lower().startswith("hpo") else "-"

        # Loss curves
        ax[0].plot(
            h[:, 0], h[:, 1],
            color="C0",
            ls=run_ls,
            label=f"{tag} train",
        )
        ax[0].plot(
            h[:, 0], h[:, 2],
            color="C1",
            ls=run_ls,
            label=f"{tag} validation",
        )

        # Validation ROC AUC is shown in orange.
        ax[1].plot(
            h[:, 0], h[:, 3],
            color="C1",
            ls=run_ls,
            label=tag,
        )
        ax[1].axvline(
            run["best_epoch"],
            color="C1",
            ls=":",
            lw=1,
            alpha=0.7,
        )

    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("BCE loss")
    ax[0].set_title(
        "loss (blue = train, orange = validation; dashed = HPO)"
    )
    ax[0].legend(fontsize=8)

    ax[1].axhline(
        0.5,
        color="gray",
        ls=":",
        label="chance (ROC AUC = 0.5)",
    )
    ax[1].set_xlabel("epoch")
    ax[1].set_title("validation ROC AUC")
    ax[1].legend(fontsize=8)

    plt.tight_layout()

    if path:
        fig.savefig(path, dpi=150)

    plt.show()

    for tag, run in runs.items():
        h = run["history"]
        lowest_loss_epoch = int(np.argmin(h[:, 2])) + 1

        print(
            f"{tag}: best ROC AUC at epoch {run['best_epoch']}, "
            f"lowest validation loss at epoch {lowest_loss_epoch}"
        )

        if abs(lowest_loss_epoch - run["best_epoch"]) > 5:
            print("   Minimum validation loss and maximum ROC AUC differ by more "
                  "than five epochs.")


plot_training(
    RUNS,
    os.path.join(OUT["figures"], "training.png"),
)


In [ ]:
def plot_pr_curves(tag, parts, path=None):
    fig, axes = plt.subplots(1, len(parts), figsize=(5.0 * len(parts), 4.4),
                             squeeze=False)
    for ax, p in zip(axes[0], parts):
        for v in VARIANTS:
            s = p["scores"][(tag, v)]
            pr, rc, _ = precision_recall_curve(p["y"], s)
            ax.plot(rc, pr, lw=1.4,
                    label=f"{v}  PR-AUC={auc(rc, pr):.3f}")
        ax.axhline(p["y"].mean(), color="gray", ls=":")
        ax.set_ylim(0, 1)
        ax.set_title(f"{p['name']}  ({p['y'].mean()*100:.1f} percent damaged)",
                     fontsize=9)
        ax.set_xlabel("recall")
        ax.legend(fontsize=7)
    axes[0][0].set_ylabel("precision")
    fig.suptitle(f"run: {tag}", y=1.02)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


for tag in RUNS:
    plot_pr_curves(tag, val_parts,
                   os.path.join(OUT["figures"], f"validation_pr_curves_{tag}.png"))


In [ ]:
def plot_error_map(tag, part, variant, path=None):
    """Show spatial probabilities, local error rates, and error types."""

    score = part["scores"][(tag, variant)]
    threshold = THRESHOLDS[tag][variant]
    y_true = part["y"]

    pred = (score >= threshold).astype(int)

    false_positive = (pred == 1) & (y_true == 0)
    false_negative = (pred == 0) & (y_true == 1)
    wrong = false_positive | false_negative

    overall_error = wrong.mean()
    fp_rate = false_positive.sum() / max((y_true == 0).sum(), 1)
    fn_rate = false_negative.sum() / max((y_true == 1).sum(), 1)

    lon = part["lon"]
    lat = part["lat"]

    # Correct geographical aspect ratio at the region's latitude.
    aspect = 1 / np.cos(np.radians(lat.mean()))

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 4.5),
        constrained_layout=True,
    )

    # Panel 1: average predicted probability in spatial bins.
    probability_map = axes[0].hexbin(
        lon,
        lat,
        C=score,
        reduce_C_function=np.mean,
        gridsize=(90, 18),
        mincnt=1,
        cmap="RdYlGn_r",
        vmin=0,
        vmax=1,
        linewidths=0,
        rasterized=True,
    )

    fig.colorbar(
        probability_map,
        ax=axes[0],
        label="mean predicted damage probability",
        shrink=0.85,
    )

    axes[0].set_title("Predicted probability")

    # Panel 2: proportion of incorrect predictions in each spatial bin.
    error_map = axes[1].hexbin(
        lon,
        lat,
        C=wrong.astype(float),
        reduce_C_function=np.mean,
        gridsize=(90, 18),
        mincnt=1,
        cmap="magma",
        vmin=0,
        vmax=1,
        linewidths=0,
        rasterized=True,
    )

    fig.colorbar(
        error_map,
        ax=axes[1],
        label="local error rate",
        shrink=0.85,
    )

    axes[1].set_title("Spatial error concentration")

    # Panel 3: distinguish false positives from false negatives.
    axes[2].scatter(
        lon,
        lat,
        s=2,
        color="lightgray",
        alpha=0.35,
        linewidths=0,
        rasterized=True,
        label="all buildings",
    )

    axes[2].scatter(
        lon[false_positive],
        lat[false_positive],
        s=12,
        marker="x",
        color="#D73027",
        alpha=0.7,
        linewidths=0.7,
        rasterized=True,
        label=f"false positive ({false_positive.sum():,})",
    )

    axes[2].scatter(
        lon[false_negative],
        lat[false_negative],
        s=12,
        marker="x",
        color="#2166AC",
        alpha=0.7,
        linewidths=0.7,
        rasterized=True,
        label=f"false negative ({false_negative.sum():,})",
    )

    axes[2].set_title(
        "Error types\n"
        f"FP rate among intact: {fp_rate:.1%} | "
        f"FN rate among damaged: {fn_rate:.1%}"
    )
    axes[2].legend(loc="best", fontsize=8, markerscale=1.4)

    for ax in axes:
        ax.set_aspect(aspect)
        ax.set_xlabel("longitude")
        ax.set_ylabel("latitude")
        ax.tick_params(labelsize=8)

    fig.suptitle(
        f"{part['name']}: {tag} / {variant}\n"
        f"validation threshold = {threshold:.3f} | "
        f"overall error = {overall_error:.1%}",
        fontsize=12,
    )

    if path:
        fig.savefig(path, dpi=180, bbox_inches="tight")

    plt.show()


plot_error_map(
    FINAL_TAG,
    val_parts[0],
    FINAL_VARIANT,
    os.path.join(
        OUT["figures"],
        "validation_error_map.png",
    ),
)


### Validation behaviour as the war goes on

The validation rows are the same spatial band at three assessment dates. The
buildings never change, so anything that moves is the imagery, the labels, or
the model's fitness for a city that is progressively more damaged.

Those buildings are outside the training band at every date, but the rows are
not independent of each other, since a
building damaged in May is still damaged in September. Read them as a trend
line, not as three separate experiments.

PR AUC depends on damage prevalence, while ROC AUC is insensitive
to prevalence. Both are shown to separate these two effects.


In [ ]:
def plot_metrics_over_time(tag, parts, path=None):
    dated = sorted([p for p in parts if "@" in p["name"]],
                   key=lambda q: q["name"].split("@")[1])
    if len(dated) < 2:
        print("fewer than two dated validation parts, nothing to plot over time")
        return None

    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    x = pd.to_datetime([p["name"].split("@")[1] for p in dated])
    for i, v in enumerate(VARIANTS):
        m = [binary_metrics(p["y"], p["scores"][(tag, v)], THRESHOLDS[tag][v])
             for p in dated]
        ax[0].plot(x, [q["PR_AUC"] for q in m], marker="o", color=f"C{i}", label=v)
        ax[1].plot(x, [q["ROC_AUC"] for q in m], marker="o", color=f"C{i}", label=v)
        ax[2].plot(x, [q["F1"] for q in m], marker="o", color=f"C{i}", label=v)
    ax[0].plot(x, [p["y"].mean() for p in dated], ls=":", color="gray",
               label="prevalence")
    ax[0].set_title("PR AUC"); ax[1].set_title("ROC AUC")
    ax[2].set_title("F1 at pooled-validation threshold")
    for a in ax:
        a.tick_params(axis="x", rotation=45)
        a.set_ylim(0, 1)
        a.legend(fontsize=7)
    fig.suptitle(f"Validation band scored at each assessment date - run {tag}",
                 y=1.03)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


plot_metrics_over_time(FINAL_TAG, val_parts,
                       os.path.join(OUT["figures"], "validation_metrics_over_time.png"))

print("F1 uses one threshold fitted to the pooled validation dates.")


## 9. What does the selected model use? (validation only)

Modality ablation zeroes one channel group at a time and measures the
average-precision drop for the selected pipeline's underlying CNN. Grad-CAM shows
which pixels drove that CNN's prediction. Both diagnostics run on validation,
never on the sealed test split.


In [ ]:
def modality_ablation(tag, part):
    """PR AUC after zeroing each channel group."""
    src, pos = part_source(part, 0)
    groups = {}
    for i, name in enumerate(CHANNEL_NAMES):
        key = name.split("_")[0] + " " + ("pre" if "_pre_" in name else "post")
        groups.setdefault(key, []).append(i)
    model = RUNS[tag]["model"]
    y = part["y"][pos]
    rows = {"full input": pr_auc_score(
        y, predict_probs(model, src, mu, sd, device=device,
                         batch=CONFIG["predict_batch"]))}
    for key, idx in groups.items():
        # zeroing happens inside predict_probs, per batch, instead of copying
        rows[f"without {key}"] = pr_auc_score(
            y, predict_probs(model, src, mu, sd, device=device,
                             batch=CONFIG["predict_batch"], zero_channels=idx))
    return pd.Series(rows, name="PR_AUC")


print(f"run: {FINAL_TAG}")
print(modality_ablation(FINAL_TAG, val_parts[0]).round(4).to_string())


In [ ]:
def grad_cam(model, x):
    """Which pixels drove this prediction? (from the last conv layer)"""
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    acts, grads = {}, {}
    h1 = convs[-1].register_forward_hook(
        lambda m, i, o: acts.__setitem__("v", o.detach()))
    h2 = convs[-1].register_full_backward_hook(
        lambda m, gi, go: grads.__setitem__("v", go[0].detach()))
    model.eval()
    xb = torch.from_numpy(x[None]).to(device).requires_grad_(True)
    logit = model(xb)
    model.zero_grad()
    logit.backward()
    h1.remove(); h2.remove()

    weight = grads["v"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weight * acts["v"]).sum(dim=1))[0].cpu().numpy()
    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam, float(torch.sigmoid(logit).item())


def show_grad_cam_examples(tag, part, n=4, path=None):
    """Grad-CAM for the highest and lowest scoring buildings of a region."""
    src, _ = part_source(part, 0)
    s = part["scores"][(tag, "cnn")]
    order = np.argsort(-s)
    picks = list(order[:n // 2]) + list(order[-(n - n // 2):])
    model = RUNS[tag]["model"]
    post_i = next(i for i, c in enumerate(CHANNEL_NAMES) if "_post_" in c)

    fig, axes = plt.subplots(2, len(picks), figsize=(3.0 * len(picks), 6.2))
    for col, j in enumerate(picks):
        xj = (np.asarray(src[int(j)]).astype(np.float32) - mu[0]) / sd[0]
        cam, prob = grad_cam(model, xj)
        cam_big = np.kron(cam, np.ones((xj.shape[1] // cam.shape[0],
                                        xj.shape[2] // cam.shape[1])))
        base = xj[post_i]
        axes[0, col].imshow(base, cmap="gray")
        axes[0, col].set_title(f"p={prob:.2f}  y={part['y'][j]}", fontsize=9)
        axes[1, col].imshow(base, cmap="gray")
        axes[1, col].imshow(cam_big, cmap="jet", alpha=0.45)
        for ax in (axes[0, col], axes[1, col]):
            ax.set_xticks([]); ax.set_yticks([])
    axes[0, 0].set_ylabel("post VV")
    axes[1, 0].set_ylabel("Grad-CAM")
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


show_grad_cam_examples(FINAL_TAG, val_parts[0], n=4,
                       path=os.path.join(OUT["figures"], "validation_grad_cam.png"))

## 10. One-shot final test evaluation: Gaza band

The Gaza test band measures performance on unseen ground within the development
city. The city-level holdouts in Section 11 measure geographic transfer.

Only the frozen `FINAL_TAG / FINAL_VARIANT` pipeline is evaluated here, using
the threshold fitted on validation. The first complete evaluation saves its
metrics, precision-recall curves and confusion matrices before writing a
completion marker. Later runs load those saved results without reconstructing
test labels or recalculating diagnostics.


In [ ]:
# The in-city test set is evaluated only after validation selection.
final_metrics_path = os.path.join(OUT["metrics"], "final_test_metrics.csv")
final_complete_path = os.path.join(OUT["metrics"], "final_test_complete.json")
legacy_test_path = os.path.join(OUT["metrics"], "test_metrics.csv")

final_offsets = OFFSETS if FINAL_VARIANT == "xgb_spatial_temporal" else [0]
method_name = f"{FINAL_TAG} / {FINAL_VARIANT}"
already_sealed = os.path.exists(final_complete_path)
test_parts = []
selected_test_scores = {}

if already_sealed:
    with open(final_complete_path) as fh:
        final_complete = json.load(fh)
    if final_complete.get("selection") != FINAL_SELECTION:
        raise RuntimeError("Final-test marker belongs to a different selection.")
    if not os.path.exists(final_metrics_path):
        raise RuntimeError("Final-test marker exists but its metrics CSV is missing.")

    FINAL_TEST_RESULTS = pd.read_csv(
        final_metrics_path, index_col=["method", "region"])
    selected = (FINAL_TEST_RESULTS.index.get_level_values("method")
                == method_name)
    if not selected.any():
        raise RuntimeError(
            "Saved final-test metrics do not contain the frozen pipeline.")
    if not selected.all():
        FINAL_TEST_RESULTS = FINAL_TEST_RESULTS.loc[selected]
        atomic_csv_dump(FINAL_TEST_RESULTS.round(6), final_metrics_path)
    print("Final test already sealed; loaded the saved metrics.")
else:
    if os.path.exists(legacy_test_path):
        warnings.warn(
            "Earlier test metrics exist for this experiment. The current "
            "workflow is sealed from this run onward, but the test band is "
            "not a pristine confirmatory set if those results were inspected.")

    test_parts = build_parts(SPLIT["test"])
    development_ids = set().union(*[
        set(p["table"]["system:index"])
        for p in train_parts + stack_parts + val_parts])
    for part in test_parts:
        overlap = development_ids & set(part["table"]["system:index"])
        if overlap:
            raise RuntimeError(
                f"{part['name']}: {len(overlap):,} test buildings also occur "
                "in a development split.")

    FINAL_TEST_SCORES = {
        FINAL_TAG: score_all(
            RUNS[FINAL_TAG], test_parts, final_offsets,
            cache_group="final_test_locked")
    }

    test_rows = {}
    for part in test_parts:
        score = variant_scores(
            FINAL_TAG, part, FINAL_VARIANT, FINAL_TEST_SCORES)
        selected_test_scores[part["name"]] = score
        m = binary_metrics(part["y"], score, FINAL_THRESHOLD)

        # Post-hoc threshold diagnostic, as in the holdout-city section: the
        # best F1 this region could reach if its threshold were fitted on
        # its own labels. Separates threshold miscalibration (frozen F1 low,
        # oracle F1 high) from weak ranking (both low). Not a valid reported
        # score on its own; the frozen-threshold F1 above stays authoritative.
        if 0 < part["y"].sum() < len(part["y"]):
            t_oracle, f1_oracle = best_f1_threshold(part["y"], score)
        else:
            t_oracle, f1_oracle = np.nan, np.nan
        m["F1_oracle"] = float(f1_oracle)
        m["thresh_oracle"] = float(t_oracle)
        test_rows[(method_name, part["name"])] = m

    FINAL_TEST_RESULTS = pd.DataFrame(test_rows).T
    FINAL_TEST_RESULTS.index.names = ["method", "region"]
    atomic_csv_dump(FINAL_TEST_RESULTS.round(6), final_metrics_path)

    fig, axes = plt.subplots(
        1, len(test_parts), figsize=(5.0 * len(test_parts), 4.4),
        squeeze=False)
    for ax, part in zip(axes[0], test_parts):
        score = selected_test_scores[part["name"]]
        precision, recall, _ = precision_recall_curve(part["y"], score)
        pr_auc = auc(recall, precision)
        ax.plot(recall, precision, lw=1.6, label=f"PR-AUC={pr_auc:.3f}")
        ax.axhline(part["y"].mean(), color="gray", ls=":",
                   label="damaged prevalence")
        ax.set_title(part["name"], fontsize=9)
        ax.set_xlabel("recall")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
    axes[0][0].set_ylabel("precision")
    fig.suptitle(f"Final test: {method_name}", y=1.02)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUT["figures"], "final_test_pr_curves.png"),
        dpi=150, bbox_inches="tight")
    plt.show()

print(FINAL_TEST_RESULTS[
    [c for c in ["n", "prevalence", "PR_AUC", "ROC_AUC", "P", "R", "F1", "F1_oracle"]
     if c in FINAL_TEST_RESULTS.columns]
].round(4).to_string())
print("F1_oracle is a post-hoc threshold diagnostic only, not a reported score.")


In [ ]:
def evaluate_final_test_confusion(
        parts, scores_by_part, threshold, tag, variant, path=None):
    """Save confusion matrices for the frozen pipeline."""
    rows = []
    fig, axes = plt.subplots(
        1, len(parts), figsize=(4.8 * len(parts), 4.5),
        squeeze=False, constrained_layout=True)
    displays = []

    for ax, part in zip(axes[0], parts):
        y_true = part["y"]
        score = scores_by_part[part["name"]]
        y_pred = (score >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        sensitivity = tp / max(tp + fn, 1)
        specificity = tn / max(tn + fp, 1)
        precision = tp / max(tp + fp, 1)
        npv = tn / max(tn + fn, 1)
        accuracy = (tp + tn) / max(cm.sum(), 1)
        rows.append({
            "region": part["name"],
            "threshold": threshold,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "accuracy": accuracy,
            "balanced_accuracy": (sensitivity + specificity) / 2,
            "sensitivity_recall": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "negative_predictive_value": npv,
        })

        display = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["intact", "damaged"])
        display.plot(
            ax=ax, cmap="Blues", colorbar=False, values_format=",")
        displays.append(display)
        date = part["name"].split("@")[-1]
        ax.set_title(
            f"{date}\nTN={tn:,}  FP={fp:,}\nFN={fn:,}  TP={tp:,}",
            fontsize=10)
        ax.set_xlabel("predicted class")
        ax.set_ylabel("true class")

    if displays:
        fig.colorbar(
            displays[-1].im_, ax=axes.ravel().tolist(),
            label="number of buildings", shrink=0.82)
    fig.suptitle(
        f"Final test confusion matrices: {tag} / {variant}\n"
        f"validation threshold = {threshold:.3f}",
        fontsize=12)
    if path:
        fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return pd.DataFrame(rows)


confusion_path = os.path.join(
    OUT["metrics"], "final_test_confusion_matrix.csv")
confusion_sealed = os.path.exists(final_complete_path)
if confusion_sealed:
    if os.path.exists(confusion_path):
        FINAL_TEST_CONFUSION = pd.read_csv(confusion_path)
        print("Loaded the sealed final-test confusion statistics.")
    else:
        FINAL_TEST_CONFUSION = None
        warnings.warn(
            "The sealed experiment predates saved confusion statistics. "
            "They were not recalculated from test labels.")
else:
    FINAL_TEST_CONFUSION = evaluate_final_test_confusion(
        test_parts, selected_test_scores, FINAL_THRESHOLD,
        FINAL_TAG, FINAL_VARIANT,
        path=os.path.join(
            OUT["figures"], "final_test_confusion_matrix.png"))
    atomic_csv_dump(FINAL_TEST_CONFUSION, confusion_path, index=False)
    atomic_json_dump({
        "selection": FINAL_SELECTION,
        "regions": [part["name"] for part in test_parts],
        "metrics_file": os.path.basename(final_metrics_path),
        "confusion_file": os.path.basename(confusion_path),
        "completed_utc": time.strftime(
            "%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, final_complete_path)
    print(f"Sealed final-test results in {final_complete_path}.")

if FINAL_TEST_CONFUSION is not None:
    print(FINAL_TEST_CONFUSION.round(4).to_string(index=False))

## 10a. Prevalence-adaptive thresholding (exploratory, not part of the frozen pipeline)

The frozen threshold above is fitted once on validation (about 60 percent
damaged) and applied everywhere. Precision, and therefore F1, is a function
of prevalence even when the model's ranking is unchanged, so a fixed
threshold drifts out of calibration wherever the true damage rate differs
from validation's. This section checks, using Gaza's own test dates as a
backtest, whether unlabelled scores alone are enough to detect and correct
for that drift.

Three predictions are compared against the same frozen CNN/stacker scores:

* **frozen** - the validation threshold, unchanged (today's default).
* **naive** - a one-shot correction using the raw mean predicted probability
  as the prevalence estimate (Classify and Count). This estimator is known
  to be biased.
* **EM** - Saerens et al. (2002) iterative prevalence correction, which
  refines the naive estimate against its own adjusted probabilities until
  convergence.

An **oracle** row (threshold fitted directly on the test labels) is included
only as an upper-bound reference. The true test labels are otherwise used
only to score the result, never to pick a threshold or a prevalence
estimate - exactly as if they were unknown, which is the point of the
exercise. This cell is diagnostic only: it does not change
`FINAL_THRESHOLD`, `final_selection.json`, or anything sealed above.

In [ ]:
def prior_correct(raw_scores, pi_source, pi_target):
    """Bayes prior-shift correction of a probability to a new class prior.

    raw_scores are P(damaged | x) under the source prior pi_source. Returns
    P(damaged | x) under pi_target, holding the model's likelihood ratio
    fixed. Monotonic in raw_scores, so it never changes ranking: ROC_AUC and
    PR_AUC are identical to the uncorrected scores.
    """
    ratio_pos = pi_target / pi_source
    ratio_neg = (1 - pi_target) / (1 - pi_source)
    numerator = ratio_pos * raw_scores
    return numerator / (numerator + ratio_neg * (1 - raw_scores))


def implied_raw_threshold(pi_source, pi_target):
    """The raw-score cutoff equivalent to adjusted-probability >= 0.5."""
    ratio_pos = pi_target / pi_source
    ratio_neg = (1 - pi_target) / (1 - pi_source)
    return ratio_neg / (ratio_pos + ratio_neg)


def em_quantify(raw_scores, pi_source, max_iter=100, tol=1e-6):
    """Saerens et al. (2002): estimate an unlabelled population's prevalence.

    Starts from the naive Classify-and-Count estimate (the raw score mean)
    and iterates the prior correction against its own output until the
    estimate stops moving. Returns the converged estimate and iteration
    count; never touches labels.
    """
    pi_target = float(np.clip(raw_scores.mean(), 1e-6, 1 - 1e-6))
    for iteration in range(1, max_iter + 1):
        adjusted = prior_correct(raw_scores, pi_source, pi_target)
        new_pi_target = float(np.clip(adjusted.mean(), 1e-6, 1 - 1e-6))
        if abs(new_pi_target - pi_target) < tol:
            return new_pi_target, iteration
        pi_target = new_pi_target
    return pi_target, max_iter


# Reuse the sealed test scores. Rebuild them from the prediction cache if
# this cell runs after a reload where the sealing cell above took the
# already-sealed branch and never populated selected_test_scores/test_parts.
if not selected_test_scores:
    test_parts = build_parts(SPLIT["test"])
    _reloaded_scores = {FINAL_TAG: score_all(
        RUNS[FINAL_TAG], test_parts, final_offsets,
        cache_group="final_test_locked")}
    for part in test_parts:
        selected_test_scores[part["name"]] = variant_scores(
            FINAL_TAG, part, FINAL_VARIANT, _reloaded_scores)

PI_TRAIN = float(np.concatenate([p["y"] for p in val_parts]).mean())
print(f"validation prevalence (correction source): {PI_TRAIN:.4f}\n")

rows = []
for part in test_parts:
    y_true = part["y"]
    scores = selected_test_scores[part["name"]]
    true_prevalence = float(y_true.mean())

    naive_pi = float(np.clip(scores.mean(), 1e-6, 1 - 1e-6))
    em_pi, em_iters = em_quantify(scores, PI_TRAIN)
    oracle_threshold, _ = best_f1_threshold(y_true, scores)

    thresholds = {
        "frozen": FINAL_THRESHOLD,
        "naive": implied_raw_threshold(PI_TRAIN, naive_pi),
        "EM": implied_raw_threshold(PI_TRAIN, em_pi),
        "oracle (uses labels)": oracle_threshold,
    }
    estimated_prevalence = {
        "frozen": PI_TRAIN, "naive": naive_pi, "EM": em_pi,
        "oracle (uses labels)": true_prevalence,
    }

    for method, threshold in thresholds.items():
        m = binary_metrics(y_true, scores, threshold)
        rows.append({
            "region": part["name"], "true_prevalence": true_prevalence,
            "method": method,
            "estimated_prevalence": estimated_prevalence[method],
            "threshold": threshold, "P": m["P"], "R": m["R"], "F1": m["F1"],
        })
    print(f"{part['name']}: true prevalence {true_prevalence:.4f}, "
          f"naive estimate {naive_pi:.4f}, "
          f"EM estimate {em_pi:.4f} ({em_iters} iterations)")

PREVALENCE_ADAPTIVE_RESULTS = pd.DataFrame(rows)
atomic_csv_dump(
    PREVALENCE_ADAPTIVE_RESULTS.round(4),
    os.path.join(OUT["metrics"], "prevalence_adaptive_threshold_diagnostic.csv"),
    index=False)
print()
print(PREVALENCE_ADAPTIVE_RESULTS.round(4).to_string(index=False))

## 11. City-level holdouts

The Gaza test band measures unseen ground within the development city. This
section applies the same frozen model, variant and validation threshold to the
registered holdout cities without refitting.

PR AUC and ROC AUC describe ranking performance without fixing an
operating threshold. Precision, recall and F1 use the threshold fitted on Gaza
validation and are therefore sensitive to prevalence and calibration changes.

`F1_oracle` is a post-hoc diagnostic that fits a threshold to each holdout
city's own labels. It can distinguish threshold mismatch from weak ranking, but
it is not a valid reported model score. Each holdout city is sealed separately,
so newly preprocessed cities can be added without recalculating completed rows.


In [ ]:
# ---------------------------------------------------------------------------
# The holdout cities. Same frozen pipeline, same frozen threshold, cities the
# model has never seen in any form.
#
# Results are sealed per city. Registering a new holdout city in
# CITY_REGISTRY later evaluates only that city and leaves the existing rows
# untouched. A city already in holdout_metrics.csv is never rescored.
# ---------------------------------------------------------------------------
holdout_metrics_path = os.path.join(OUT["metrics"], "holdout_metrics.csv")
holdout_complete_path = os.path.join(OUT["metrics"], "holdout_complete.json")

HOLDOUT_RESULTS = None
holdout_scores = {}
skipped = {}

# Build one entry at a time: a holdout city that notebook 1 has not
# preprocessed yet raises inside load_city, and that is a missing input
# rather than a reason to discard results for cities that are ready.
holdout_parts = []
for entry in SPLIT.get("holdout", []):
    try:
        holdout_parts += build_parts([entry])
    except (FileNotFoundError, KeyError) as exc:
        skipped[entry] = f"{type(exc).__name__}: {exc}"

if not holdout_parts and not skipped:
    print("no holdout cities registered - nothing to do")
else:
    for p in holdout_parts:
        if city_role(p["city"]) != "holdout":
            raise RuntimeError(
                f"{p['name']}: city {p['city']} is role="
                f"{city_role(p['city'])}, not holdout. The holdout split part "
                "must contain only cities held out of development.")

    previous = None
    if os.path.exists(holdout_metrics_path):
        previous = pd.read_csv(holdout_metrics_path,
                               index_col=["method", "region"])
        if os.path.exists(holdout_complete_path):
            with open(holdout_complete_path) as fh:
                if json.load(fh).get("selection") != FINAL_SELECTION:
                    raise RuntimeError(
                        "Saved holdout metrics belong to a different frozen "
                        "selection. Use a new experiment_name.")
        selected = (previous.index.get_level_values("method")
                    == method_name)
        if not selected.all():
            previous = previous.loc[selected]
            atomic_csv_dump(previous.round(6), holdout_metrics_path)
    done_regions = (set(previous.index.get_level_values("region"))
                    if previous is not None else set())

    todo = [p for p in holdout_parts if p["name"] not in done_regions]
    print(f"holdout cities: {[p['name'] for p in holdout_parts]}")
    if done_regions:
        print(f"already scored, kept as-is: {sorted(done_regions)}")

    new_rows = {}
    for p in todo:
        try:
            scores = score_all(RUNS[FINAL_TAG], [p], final_offsets,
                               cache_group="holdout_locked")
        except FileNotFoundError as exc:
            skipped[p["name"]] = f"FileNotFoundError: {exc}"
            continue

        # score_part swallows a missing patch array into NaN, so an entirely
        # unscored city arrives here as all-NaN rather than as an exception.
        # Metrics would be meaningless for an entirely unscored city.
        if not np.isfinite(scores[p["name"]][0]).any():
            skipped[p["name"]] = "no usable patches at the assessment date"
            continue

        score = variant_scores(FINAL_TAG, p, FINAL_VARIANT, {FINAL_TAG: scores})
        holdout_scores[p["name"]] = score

        m = binary_metrics(p["y"], score, FINAL_THRESHOLD)

        # Post-hoc threshold diagnostic. It is stored separately from the
        # metrics that use the frozen Gaza validation threshold.
        if 0 < p["y"].sum() < len(p["y"]):
            t_oracle, f1_oracle = best_f1_threshold(p["y"], score)
        else:
            t_oracle, f1_oracle = np.nan, np.nan
        m["F1_oracle"] = float(f1_oracle)
        m["thresh_oracle"] = float(t_oracle)
        new_rows[(method_name, p["name"])] = m

    if new_rows:
        fresh = pd.DataFrame(new_rows).T
        fresh.index.names = ["method", "region"]
        HOLDOUT_RESULTS = (pd.concat([previous, fresh])
                           if previous is not None else fresh)
        atomic_csv_dump(HOLDOUT_RESULTS.round(6), holdout_metrics_path)
        atomic_json_dump({
            "selection": FINAL_SELECTION,
            "regions": sorted(set(
                HOLDOUT_RESULTS.index.get_level_values("region"))),
            "metrics_file": os.path.basename(holdout_metrics_path),
            "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, holdout_complete_path)
    else:
        HOLDOUT_RESULTS = previous

    if skipped:
        print("not evaluated because required notebook 1 patches are missing:")
        for name, why in skipped.items():
            print(f"   {name}: {why}")

    if HOLDOUT_RESULTS is None or not len(HOLDOUT_RESULTS):
        print("no holdout city could be scored yet")
    else:
        cols = [c for c in ["n", "prevalence", "PR_AUC", "ROC_AUC",
                            "P", "R", "F1", "F1_oracle"]
                if c in HOLDOUT_RESULTS.columns]
        print()
        print(f"Holdout cities: {method_name}, threshold "
              f"{FINAL_THRESHOLD:.4f} fitted on Gaza validation:")
        print(HOLDOUT_RESULTS[cols].round(4).to_string())
        print()
        print("P, R and F1 use the threshold fitted on Gaza validation.")
        print("F1_oracle is a post-hoc threshold diagnostic only.")


## Experiment controls

A new scientific configuration requires a new `experiment_name`. Completed
CNN checkpoints, prediction caches and compatible stackers are reused within
an experiment. An unfinished Optuna study resumes from `optuna.db`; increasing
`hpo.n_trials` before final selection runs only the additional trials.

Set `EVALUATE_ONLY = True` to load completed CNNs and stackers without fitting.
Normalization statistics are restored from the checkpoint, and the frozen
selection continues to determine the model, prediction variant and threshold.

The main optional controls are:

* `hpo.enabled`: enables or disables the Optuna search.
* `stacking.enabled`: enables the learned spatial second stage.
* `temporal_stacking.enabled`: adds temporal score features to the stacker.
* `PREP["temporal"]` in `pipeline.py`: defines the temporal offset dates and
  the number of Sentinel-1 acquisitions used for those offset composites.
* `"@*"` in a split entry: expands that band to all registered assessment
  dates.

Holdout cities come from `CITY_REGISTRY` entries with `role: "holdout"`.
After notebook 1 has created their patches, an evaluation-only run scores
holdouts that do not yet have sealed results. `check_split_holdout()` prevents
holdout cities from appearing in development bands.
